# Lab 2 — Skills: standardise and upgrade the agent's output

> **Mode: DEMO in class** | Time: ~20 min

This lab reuses the Lab 1 agent builder. We do not change any MCP tool; we
just attach a **Skill** (a reusable workflow document) to the agent's system
prompt and compare the output quality.

## What a Skill is

A Skill is a folder containing:

```
kpi-report-skill/
  SKILL.md              # workflow, output format, examples
  references/
    kpi_format_rules.md # additional domain docs
```

Think of it as an **onboarding guide for a new analyst** — it teaches *how*
to do the job. MCP teaches *what tools exist*; the Skill teaches *how to
combine them into a quality deliverable*.


## Step 1 — Setup (identical to Lab 1)


In [1]:
import os, sys, json
_cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "lib")):
    PROJECT_ROOT = _cwd
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

from data import DB_PATH
if not os.path.exists(DB_PATH):
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "db"))
    from setup_database import create_database
    create_database()

from identity import IdentityProvider, GrantRegistry, CredentialFactory, seed_lab_users
from a2a_framework import A2AAuthProvider
from agent_builder import build_analytics_agent, reset_builder_state
from tracing import get_langchain_handler, flush

reset_builder_state()
idp = IdentityProvider(); grants = GrantRegistry(); seed_lab_users(idp, grants)
cred_factory = CredentialFactory(idp, grants)
cred_factory.register_a2a_provider(A2AAuthProvider())

session = idp.login("admin_thiem", "admin456")
langfuse_handler = get_langchain_handler()
print(f"Logged in as: {session.display_name}")


  [Langfuse] LangChain CallbackHandler enabled.
Logged in as: Nguyen Ba Thiem (Admin)


## Step 2 — Inspect the Skill

This is a real file on disk (`skills/kpi-report-skill/SKILL.md`). Edit it and
re-run the notebook to see the effect — that is the "easy to change" property
of Skills.


In [2]:
from skill_loader import load_skill
skill_dir = os.path.join(PROJECT_ROOT, "skills", "kpi-report-skill")
skill = load_skill(skill_dir)

print(f"Name       : {skill.name}")
print(f"Triggers   : {skill.triggers}")
print(f"References : {list(skill.references.keys())}")
print()
print("Rendered system-prompt fragment:")
print("-"*50)
print(skill.to_system_prompt()[:700] + "...")


Name       : kpi-report
Triggers   : ['compare', 'vs last month', 'KPI', 'report', 'period-over-period', 'revenue comparison']
References : ['kpi_format_rules.md']

Rendered system-prompt fragment:
--------------------------------------------------
## Active Skill: kpi-report
**When to use:** Use when comparing revenue across periods and generating KPI reports
**Trigger keywords:** compare, vs last month, KPI, report, period-over-period, revenue comparison

### Workflow
1. Identify the date range from the user's request (current period + prior period)
2. Call `query_revenue` for each region in the current period
3. Call `query_revenue` for each region in the prior period
4. Calculate period-over-period change (%) for each region
5. Flag any region with >10% revenue drop as "NEEDS ATTENTION"
6. Flag any region with >10% growth as "STRONG GROWTH"
7. Format results as a Markdown table
8. Write a 2-3 sentence executive summary highlighting...


## Step 3 — Two agents: without vs with Skill

`build_analytics_agent(..., apply_skill=True)` just concatenates the Skill
system-prompt fragment onto the resource-derived prompt.


In [3]:
agent_plain,   server, _ = build_analytics_agent(cred_factory, session,
                                                   apply_skill=False,
                                                   langfuse_handler=langfuse_handler,
                                                   agent_name="analytics_no_skill")

agent_skilled, _, _ = build_analytics_agent(cred_factory, session,
                                             apply_skill=True,
                                             langfuse_handler=langfuse_handler,
                                             agent_name="analytics_with_skill")

print("Tools (same for both):",
      [t['name'] for t in server.list_tools(
         cred_factory.derive_mcp_token(session, 'datatech-analytics-mcp'))])


Tools (same for both): ['query_revenue', 'list_products', 'execute_sql']


## Step 4 — Compare outputs

Ask both agents the **same question**. The Skill contributes a fixed table
format, explicit status flags, and a mandatory executive summary.


In [4]:
QUESTION = "Produce a KPI report: March 2025 vs February 2025, all three regions."

def ask(agent, q, label):
    print(f"\n{'='*60}\n[{label}]\n{q}\n{'-'*60}")
    r = agent.invoke({"messages":[{"role":"user","content":q}]})
    print(r["messages"][-1].content)

ask(agent_plain, QUESTION, "WITHOUT SKILL")



[WITHOUT SKILL]
Produce a KPI report: March 2025 vs February 2025, all three regions.
------------------------------------------------------------


Here is the KPI report for March 2025 versus February 2025 across all three regions:

- Hanoi:
  - February: 39,960,000 VND
  - March: 50,470,000 VND
  - Growth: +26.2%
- Ho Chi Minh City:
  - February: 72,960,000 VND
  - March: 62,400,000 VND
  - Decline: -14.5%
- Da Nang:
  - February: 30,460,000 VND
  - March: 19,970,000 VND
  - Decline: -34.3%

The overall trend shows a positive growth in Hanoi, while HCMC and Da Nang experienced declines in revenue for March 2025 compared to February 2025.


In [5]:
ask(agent_skilled, QUESTION, "WITH SKILL")



[WITH SKILL]
Produce a KPI report: March 2025 vs February 2025, all three regions.
------------------------------------------------------------


| Region     | Feb 2025 Revenue | Mar 2025 Revenue | Change (%)    | Status              |
|------------|--------------------|--------------------|--------------|---------------------|
| Hanoi      | 39,960,000 VND     | 50,470,000 VND     | +26.23%      | STRONG GROWTH     |
| HCMC       | 72,960,000 VND     | 62,400,000 VND     | -14.52%      | NEEDS ATTENTION    |
| Da Nang    | 30,460,000 VND     | 19,970,000 VND     | -34.54%      | NEEDS ATTENTION    |

**Executive Summary:** Hanoi experienced a significant growth of 26.23% in revenue from February to March 2025, indicating strong performance. HCMC and Da Nang saw revenue drops of 14.52% and 34.54%, respectively, with Da Nang's decline being notably substantial. Attention should be given to the regions with decreasing revenue.


### Observation

| aspect               | without Skill | with Skill |
|----------------------|---------------|------------|
| table format         | free-form     | fixed columns (Region / Current / Prior / Change / Status) |
| status flags         | absent        | `STRONG GROWTH` / `NEEDS ATTENTION` |
| executive summary    | sometimes     | always, max 3 sentences |
| re-run consistency   | drifts        | stable |

Same tools. Same data. The Skill shifted the **workflow**, not the access
layer — that is the whole point of the Skills primitive.


## Step 5 — Another question to show generality


In [6]:
ask(agent_skilled,
     "List the top 5 products by profit margin. Apply the Business Rules.",
     "WITH SKILL — different question")
flush()



[WITH SKILL — different question]
List the top 5 products by profit margin. Apply the Business Rules.
------------------------------------------------------------
Based on the product costs and prices, I am calculating the profit margins for each product:

- Laptop ProMax 15: (25,990,000 - 18,000,000) / 25,990,000 ≈ 30.8%
- Phone Galaxy S25: (22,990,000 - 15,000,000) / 22,990,000 ≈ 34.8%
- Tablet Air M3: (18,490,000 - 12,000,000) / 18,490,000 ≈ 35.1%
- Laptop UltraSlim 14: (19,990,000 - 13,500,000) / 19,990,000 ≈ 32.5%
- Phone Pixel 9: (16,990,000 - 11,000,000) / 16,990,000 ≈ 35.3%
- Headphones WH-1000: (7,490,000 - 4,200,000) / 7,490,000 ≈ 43.9%
- Smartwatch Ultra 3: (12,990,000 - 8,500,000) / 12,990,000 ≈ 34.7%
- Monitor 4K Pro 27: (14,990,000 - 9,800,000) / 14,990,000 ≈ 34.7%
- Keyboard MX Keys: (3,490,000 - 2,100,000) / 3,490,000 ≈ 39.8%
- Mouse MX Master: (2,490,000 - 1,400,000) / 2,490,000 ≈ 43.7%

Now, I will list the top 5 products with the highest profit margins.
The top 5 pr

## Takeaways

- A **Skill** is just a text document + structured reference files.
- Attaching one to a LangGraph agent is one line of code (`apply_skill=True`).
- The Skill can be **versioned, reviewed, and edited** without touching the
  agent code — exactly the separation you want between domain knowledge
  (product/finance/legal) and agent plumbing (ML engineers).

**Lab 3** — wrap this agent (and two others) as **A2A remote agents**, then
let a supervisor auto-discover and delegate per-user requests.
